# LeetCode #177 — Nth Highest Salary: hàm SQL có tham số, `OFFSET N−1` và `DENSE_RANK`

**Mức độ:** Medium · **Chủ đề:** Database (SQL), Pandas

Skeleton trên LeetCode:

```sql
-- MySQL
CREATE FUNCTION getNthHighestSalary(N INT) RETURNS INT
BEGIN
  RETURN (
      # Write your MySQL query statement below.
  );
END
```

```python
# Pandas
import pandas as pd

def nth_highest_salary(employee: pd.DataFrame, N: int) -> pd.DataFrame:
```

Mục tiêu của notebook:

1. Tổng quát hóa #176: lương cao thứ `N` khác nhau = bỏ `N − 1` mức đầu (`OFFSET N − 1`).
2. Viết **hàm** trong MySQL (`CREATE FUNCTION`, `SET`, `RETURN`) và vì sao phải `SET N = N − 1` trước.
3. Ba cách: `LIMIT/OFFSET`, `DENSE_RANK()` (hàm cửa sổ), truy vấn con **tương quan** (đếm số mức lương lớn hơn).
4. Trường hợp biên: `N` lớn hơn số mức lương, `N <= 0`.

# 1. Bài toán là gì?

Bảng `Employee(id int, salary int)`, `id` là khóa chính.

Viết hàm trả về mức lương cao thứ `N` **khác nhau**. Nếu có ít hơn `N` mức lương khác nhau, trả về `null`.

(Bản Pandas trả về `DataFrame` một dòng, tên cột là `getNthHighestSalary(N)` với `N` là số cụ thể, ví dụ `getNthHighestSalary(2)`.)

---

## Ví dụ

```text
Employee = [(1, 100), (2, 200), (3, 300)], N = 2   →  getNthHighestSalary(2) = 200
Employee = [(1, 100)],                     N = 2   →  getNthHighestSalary(2) = null
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. SQL cơ bản — truy vấn bảng dữ liệu
2. Hàm cửa sổ (window function) trong SQL
3. Pandas DataFrame — SQL bằng Python

Cuối mục có phần **2.4 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 SQL cơ bản — truy vấn bảng dữ liệu

### Bảng, hàng, cột

Cơ sở dữ liệu quan hệ lưu dữ liệu trong các **bảng**. Mỗi **hàng** là một bản ghi, mỗi **cột** là một thuộc tính có kiểu (số, chuỗi…). Ô có thể **rỗng** — giá trị đặc biệt `NULL` ("không biết / không có").

```text
Employee
+----+-------+--------+--------------+
| id | name  | salary | departmentId |
+----+-------+--------+--------------+
| 1  | Joe   | 70000  | 1            |
| 2  | Jim   | 90000  | 1            |
| 3  | Henry | 80000  | 2            |
+----+-------+--------+--------------+
```

Trong notebook ta chạy SQL thật bằng **`sqlite3`** (có sẵn trong Python, chạy trong bộ nhớ). LeetCode dùng MySQL — cú pháp các câu cơ bản giống nhau, chỗ khác biệt sẽ được ghi chú.

In [1]:
import sqlite3

def sql_demo(query, params=()):
    con = sqlite3.connect(":memory:")
    con.execute("CREATE TABLE Employee (id INTEGER, name TEXT, salary INTEGER, departmentId INTEGER)")
    con.executemany("INSERT INTO Employee VALUES (?, ?, ?, ?)",
                    [(1, "Joe", 70000, 1), (2, "Jim", 90000, 1), (3, "Henry", 80000, 2),
                     (4, "Sam", 60000, 2), (5, "Max", 90000, 1)])
    con.execute("CREATE TABLE Department (id INTEGER, name TEXT)")
    con.executemany("INSERT INTO Department VALUES (?, ?)", [(1, "IT"), (2, "Sales")])
    cur = con.execute(query, params)
    cols = [d[0] for d in cur.description]
    out = cur.fetchall()
    con.close()
    print(cols)
    for r in out:
        print("  ", r)

sql_demo("SELECT * FROM Employee")

['id', 'name', 'salary', 'departmentId']
   (1, 'Joe', 70000, 1)
   (2, 'Jim', 90000, 1)
   (3, 'Henry', 80000, 2)
   (4, 'Sam', 60000, 2)
   (5, 'Max', 90000, 1)


### Thứ tự **viết** và thứ tự **thực hiện** của một câu SELECT

```text
Viết:        SELECT ... FROM ... WHERE ... GROUP BY ... HAVING ... ORDER BY ... LIMIT ...
Thực hiện:   FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT
```

Hiểu thứ tự thực hiện giải thích nhiều lỗi: `WHERE` chạy **trước** khi gom nhóm nên không dùng được hàm gộp (`COUNT`, `MAX`…) trong `WHERE` — phải dùng `HAVING`.

### Lọc, sắp xếp, giới hạn, bỏ trùng

In [2]:
print("Lương > 65000, giảm dần, lấy 2 hàng:")
sql_demo("SELECT name, salary FROM Employee WHERE salary > 65000 ORDER BY salary DESC LIMIT 2")
print("Các mức lương KHÁC NHAU, giảm dần, bỏ qua 1 hàng đầu (OFFSET):")
sql_demo("SELECT DISTINCT salary FROM Employee ORDER BY salary DESC LIMIT 1 OFFSET 1")

Lương > 65000, giảm dần, lấy 2 hàng:
['name', 'salary']
   ('Jim', 90000)
   ('Max', 90000)
Các mức lương KHÁC NHAU, giảm dần, bỏ qua 1 hàng đầu (OFFSET):
['salary']
   (80000,)


### Hàm gộp và GROUP BY

`COUNT`, `SUM`, `AVG`, `MIN`, `MAX` gộp nhiều hàng thành một giá trị. `GROUP BY cột` chia bảng thành nhóm theo cột đó, gộp riêng từng nhóm. `HAVING` lọc **sau** khi gộp.

In [3]:
sql_demo("""
SELECT departmentId, COUNT(*) AS n, MAX(salary) AS top
FROM Employee
GROUP BY departmentId
HAVING COUNT(*) >= 2
""")

['departmentId', 'n', 'top']
   (1, 3, 90000)
   (2, 2, 80000)


### NULL: "không có giá trị"

- Hàm gộp trên **tập rỗng** (trừ `COUNT`) trả về `NULL` — đây là mẹo để trả về `NULL` khi "không có đáp án".
- So sánh với `NULL` bằng `=` luôn **không đúng** → phải dùng `IS NULL`.
- `IFNULL(x, d)` / `COALESCE(x, d)` thay `NULL` bằng giá trị mặc định.

In [4]:
sql_demo("SELECT MAX(salary) AS m FROM Employee WHERE salary > 1000000")
sql_demo("SELECT NULL = NULL AS bang_nhau, NULL IS NULL AS la_null, COALESCE(NULL, 0) AS mac_dinh")

['m']
   (None,)
['bang_nhau', 'la_null', 'mac_dinh']
   (None, 1, 0)


### Truy vấn con (subquery)

Một câu `SELECT` có thể nằm **bên trong** câu khác: trả về một giá trị (dùng trong `WHERE x > (...)`), một danh sách (dùng với `IN`), hoặc một bảng tạm (đặt trong `FROM`).

In [5]:
print("Lương cao thứ hai = lương lớn nhất trong số những lương nhỏ hơn lương lớn nhất:")
sql_demo("SELECT MAX(salary) AS second FROM Employee WHERE salary < (SELECT MAX(salary) FROM Employee)")

Lương cao thứ hai = lương lớn nhất trong số những lương nhỏ hơn lương lớn nhất:
['second']
   (80000,)


### JOIN: ghép hai bảng

`A JOIN B ON điều_kiện` tạo các cặp hàng (a, b) thỏa điều kiện.

- **INNER JOIN** (mặc định): chỉ giữ cặp khớp.
- **LEFT JOIN**: giữ **mọi** hàng của A; hàng không khớp thì cột của B là `NULL`.
- **Self-join**: ghép bảng với **chính nó** (đặt hai bí danh khác nhau) — để so sánh các hàng trong cùng bảng.

In [6]:
sql_demo("""
SELECT d.name AS department, e.name AS employee, e.salary
FROM Employee e JOIN Department d ON e.departmentId = d.id
ORDER BY d.name, e.salary DESC
""")
print("Self-join: cặp nhân viên cùng phòng, người trước lương thấp hơn người sau")
sql_demo("""
SELECT a.name, b.name FROM Employee a JOIN Employee b
ON a.departmentId = b.departmentId AND a.salary < b.salary
""")

['department', 'employee', 'salary']
   ('IT', 'Jim', 90000)
   ('IT', 'Max', 90000)
   ('IT', 'Joe', 70000)
   ('Sales', 'Henry', 80000)
   ('Sales', 'Sam', 60000)
Self-join: cặp nhân viên cùng phòng, người trước lương thấp hơn người sau
['name', 'name']
   ('Joe', 'Jim')
   ('Joe', 'Max')
   ('Sam', 'Henry')


### Lỗi hay gặp

- Quên `DISTINCT` khi hỏi "giá trị lớn thứ k **khác nhau**".
- Dùng hàm gộp trong `WHERE` (phải dùng `HAVING` hoặc truy vấn con).
- `LIMIT 1 OFFSET k` trên bảng **không đủ** hàng trả về **không hàng nào** (không phải `NULL`) — bọc trong truy vấn con hoặc dùng `MAX` để có `NULL`.
- So sánh `= NULL` thay vì `IS NULL`.
- Trong MySQL, `LIMIT` không nhận biểu thức (`LIMIT N-1` lỗi) → phải tính trước (`SET N = N - 1`).

## 2.2 Hàm cửa sổ (window function) trong SQL

### Hàm gộp vs hàm cửa sổ

- **Hàm gộp + GROUP BY**: mỗi nhóm **co lại** thành một hàng.
- **Hàm cửa sổ** `f(...) OVER (PARTITION BY ... ORDER BY ...)`: **giữ nguyên** mọi hàng, mỗi hàng nhận thêm một giá trị tính trên "cửa sổ" các hàng liên quan.

```text
PARTITION BY departmentId   → chia thành các nhóm (giống GROUP BY nhưng không co lại)
ORDER BY salary DESC        → thứ tự trong mỗi nhóm
```

Có từ MySQL 8.0 và SQLite 3.25.

In [7]:
import sqlite3

def window_demo(query):
    con = sqlite3.connect(":memory:")
    con.execute("CREATE TABLE Scores (id INTEGER, score REAL, dept INTEGER)")
    con.executemany("INSERT INTO Scores VALUES (?, ?, ?)",
                    [(1, 3.50, 1), (2, 3.65, 1), (3, 4.00, 2), (4, 3.85, 2), (5, 4.00, 1), (6, 3.65, 2)])
    cur = con.execute(query)
    print([d[0] for d in cur.description])
    for r in cur.fetchall():
        print("  ", r)
    con.close()

### Ba hàm đánh số: ROW_NUMBER, RANK, DENSE_RANK

Khác nhau ở cách xử lý **giá trị bằng nhau**:

| giá trị | ROW_NUMBER | RANK | DENSE_RANK |
|---|---|---|---|
| 4.00 | 1 | 1 | 1 |
| 4.00 | 2 | 1 | 1 |
| 3.85 | 3 | **3** (nhảy) | **2** (liền) |

In [8]:
window_demo("""
SELECT score,
       ROW_NUMBER() OVER (ORDER BY score DESC) AS row_num,
       RANK()       OVER (ORDER BY score DESC) AS rnk,
       DENSE_RANK() OVER (ORDER BY score DESC) AS dense
FROM Scores ORDER BY score DESC
""")

['score', 'row_num', 'rnk', 'dense']
   (4.0, 1, 1, 1)
   (4.0, 2, 1, 1)
   (3.85, 3, 3, 2)
   (3.65, 4, 4, 3)
   (3.65, 5, 4, 3)
   (3.5, 6, 6, 4)


### PARTITION BY: đánh số riêng trong từng nhóm

"Top-k mỗi nhóm" = đánh số trong nhóm rồi lọc `<= k`. Vì `WHERE` chạy **trước** hàm cửa sổ, phải bọc trong truy vấn con hoặc `WITH`.

In [9]:
window_demo("""
WITH ranked AS (
    SELECT dept, id, score,
           DENSE_RANK() OVER (PARTITION BY dept ORDER BY score DESC) AS r
    FROM Scores
)
SELECT dept, id, score FROM ranked WHERE r = 1 ORDER BY dept, id
""")

['dept', 'id', 'score']
   (1, 5, 4.0)
   (2, 3, 4.0)


### LAG / LEAD: nhìn hàng trước / sau

`LAG(col, k)` lấy giá trị của hàng cách **trước** `k` hàng (theo `ORDER BY` của cửa sổ), `LEAD` lấy hàng **sau**; không có thì `NULL`.

In [10]:
window_demo("""
SELECT id, score,
       LAG(score, 1)  OVER (ORDER BY id) AS truoc,
       LEAD(score, 1) OVER (ORDER BY id) AS sau
FROM Scores ORDER BY id
""")

['id', 'score', 'truoc', 'sau']
   (1, 3.5, None, 3.65)
   (2, 3.65, 3.5, 4.0)
   (3, 4.0, 3.65, 3.85)
   (4, 3.85, 4.0, 4.0)
   (5, 4.0, 3.85, 3.65)
   (6, 3.65, 4.0, None)


### WITH (CTE): đặt tên cho bảng tạm

`WITH ten AS (SELECT ...) SELECT ... FROM ten` giúp chia truy vấn dài thành các bước dễ đọc — tương tự gán biến trung gian trong Python.

### Lỗi hay gặp

- Dùng `RANK` khi đề muốn thứ hạng **liền nhau** (cần `DENSE_RANK`), hoặc ngược lại.
- Lọc kết quả hàm cửa sổ trong `WHERE` cùng câu → lỗi; phải bọc truy vấn con / CTE.
- `rank` là **từ khóa** trong MySQL 8 → đặt tên cột là `` `rank` `` (backtick).
- Quên `PARTITION BY` → đánh số trên **toàn bảng** thay vì từng nhóm.

## 2.3 Pandas DataFrame — SQL bằng Python

### DataFrame là gì?

`pandas.DataFrame` là một **bảng** trong bộ nhớ: các cột có tên, mỗi cột là một `Series` (mảng có nhãn). LeetCode có phiên bản Pandas cho các bài SQL; mỗi thao tác SQL có một tương đương:

| SQL | Pandas |
|---|---|
| `WHERE cond` | `df[df["col"] > x]` |
| `SELECT a, b` | `df[["a", "b"]]` |
| `ORDER BY a DESC` | `df.sort_values("a", ascending=False)` |
| `DISTINCT` | `df.drop_duplicates()` / `series.unique()` |
| `GROUP BY` + gộp | `df.groupby("k")["v"].max()` |
| `JOIN` | `df1.merge(df2, left_on=..., right_on=...)` |
| `DENSE_RANK()` | `series.rank(method="dense", ascending=False)` |
| `LAG(col)` | `series.shift(1)` |

In [11]:
import pandas as pd

emp = pd.DataFrame({"id": [1, 2, 3, 4, 5],
                    "name": ["Joe", "Jim", "Henry", "Sam", "Max"],
                    "salary": [70000, 90000, 80000, 60000, 90000],
                    "departmentId": [1, 1, 2, 2, 1]})
dep = pd.DataFrame({"id": [1, 2], "name": ["IT", "Sales"]})
print(emp)

   id   name  salary  departmentId
0   1    Joe   70000             1
1   2    Jim   90000             1
2   3  Henry   80000             2
3   4    Sam   60000             2
4   5    Max   90000             1


### Lọc, chọn cột, sắp xếp, bỏ trùng

In [12]:
print(emp[emp["salary"] > 65000][["name", "salary"]].sort_values("salary", ascending=False).head(2))
print("lương khác nhau giảm dần:", sorted(emp["salary"].unique().tolist(), reverse=True))

  name  salary
1  Jim   90000
4  Max   90000
lương khác nhau giảm dần: [90000, 80000, 70000, 60000]


### groupby và transform

- `groupby(k)[v].max()` → **co** mỗi nhóm thành một giá trị (như `GROUP BY`).
- `groupby(k)[v].transform("max")` → trả về giá trị của nhóm **cho từng hàng** (giống hàm cửa sổ `MAX() OVER (PARTITION BY k)`), rất tiện để lọc "hàng có giá trị lớn nhất nhóm".

In [13]:
print(emp.groupby("departmentId")["salary"].max())
top = emp[emp["salary"] == emp.groupby("departmentId")["salary"].transform("max")]
print(top)

departmentId
1    90000
2    80000
Name: salary, dtype: int64
   id   name  salary  departmentId
1   2    Jim   90000             1
2   3  Henry   80000             2
4   5    Max   90000             1


### merge (JOIN), rank, shift

In [14]:
m = emp.merge(dep, left_on="departmentId", right_on="id", suffixes=("", "_dep"))
print(m[["name_dep", "name", "salary"]])
s = pd.Series([4.00, 4.00, 3.85, 3.65])
print("rank dense:", s.rank(method="dense", ascending=False).tolist(), "| rank min (như RANK):", s.rank(method="min", ascending=False).tolist())
print("shift(1) (như LAG):", s.shift(1).tolist())

  name_dep   name  salary
0       IT    Joe   70000
1       IT    Jim   90000
2    Sales  Henry   80000
3    Sales    Sam   60000
4       IT    Max   90000
rank dense: [1.0, 1.0, 2.0, 3.0] | rank min (như RANK): [1.0, 1.0, 3.0, 4.0]
shift(1) (như LAG): [nan, 4.0, 4.0, 3.85]


### Lỗi hay gặp

- `df[...]` lọc trả về DataFrame **có thể rỗng** — kiểm tra `len(...)` trước khi lấy `.iloc[0]`.
- `rank` trả về số thực (`1.0`) → `astype(int)` nếu cần.
- `merge` hai bảng cùng tên cột → cột bị thêm hậu tố `_x`, `_y` (hoặc `suffixes` tự đặt).
- Khi đề yêu cầu trả `null`, trả `None` trong một DataFrame một hàng — không trả DataFrame rỗng.

## 2.4 Áp dụng vào bài này

Tổng quát #176 cho lương cao thứ `N`: `LIMIT 1 OFFSET N−1` trên các lương **khác nhau** (trong MySQL phải tính `N − 1` trước vì `LIMIT` không nhận biểu thức), hoặc dùng **`DENSE_RANK()`** đánh hạng liền nhau rồi lọc hạng `N`. Vẫn phải trả `NULL` khi không đủ `N` mức lương.

# 3. Nền tảng: chạy SQL có tham số

`sqlite3` cho phép đặt **tham số** bằng dấu `?` rồi truyền giá trị riêng — an toàn hơn ghép chuỗi (tránh SQL injection) và giống ý tưởng "biến `N`" trong hàm MySQL.

In [15]:
import sqlite3
import random
import pandas as pd


def run_sql(sql: str, tables: dict, params=()) -> pd.DataFrame:
    conn = sqlite3.connect(":memory:")
    for name, df in tables.items():
        df.to_sql(name, conn, index=False)
    out = pd.read_sql_query(sql, conn, params=params)
    conn.close()
    return out


def rows(df: pd.DataFrame) -> list:
    return [[None if pd.isna(v) else v for v in r] for r in df.itertuples(index=False)]


def emp_df(salaries) -> pd.DataFrame:
    return pd.DataFrame({"id": range(1, len(salaries) + 1), "salary": salaries})


t = {"Employee": emp_df([100, 200, 300])}
print(run_sql("SELECT salary FROM Employee WHERE salary >= ?", t, params=(200,)))

   salary
0     200
1     300


## Hàm trong MySQL

```sql
CREATE FUNCTION tên(tham_số KIỂU) RETURNS KIỂU
BEGIN
  câu lệnh;            -- ví dụ SET N = N - 1;
  RETURN ( truy vấn trả về một giá trị );
END
```

`RETURN (SELECT …)` cũng là truy vấn con **vô hướng**: không có dòng nào thì giá trị là `NULL` — đúng yêu cầu đề.

**Điểm hay bị vấp:** trong MySQL, `LIMIT` / `OFFSET` chỉ nhận **hằng số hoặc biến**, không nhận biểu thức như `N - 1`. Vì vậy phải `SET N = N - 1;` (hoặc khai báo biến `DECLARE M INT; SET M = N - 1;`) trước rồi viết `OFFSET N`.

SQLite không có `CREATE FUNCTION` kiểu này; ta mô phỏng "hàm" bằng một hàm Python chạy truy vấn với tham số `N − 1`.

# 4. Cách 1: `DISTINCT` + `ORDER BY DESC` + `LIMIT 1 OFFSET N−1`

Bản MySQL hoàn chỉnh:

```sql
CREATE FUNCTION getNthHighestSalary(N INT) RETURNS INT
BEGIN
  SET N = N - 1;
  RETURN (
      SELECT DISTINCT salary
      FROM Employee
      ORDER BY salary DESC
      LIMIT 1 OFFSET N
  );
END
```

In [16]:
SQL_OFFSET = """
SELECT (
    SELECT DISTINCT salary FROM Employee
    ORDER BY salary DESC
    LIMIT 1 OFFSET ?
) AS value
"""

def nth_by_offset(salaries, N):
    if N <= 0:
        return None                      # OFFSET âm không hợp lệ; quy ước trả null
    return rows(run_sql(SQL_OFFSET, {"Employee": emp_df(salaries)}, params=(N - 1,)))[0][0]

for sal, N in [([100, 200, 300], 2), ([100], 2), ([300, 300, 200, 100], 2), ([300, 300, 200, 100], 3), ([5, 4], 0)]:
    print(f"{str(sal):<22} N = {N}: {nth_by_offset(sal, N)}")

[100, 200, 300]        N = 2: 200
[100]                  N = 2: None
[300, 300, 200, 100]   N = 2: 200
[300, 300, 200, 100]   N = 3: 100
[5, 4]                 N = 0: None


# 5. Cách 2: hàm cửa sổ `DENSE_RANK()`

**Hàm cửa sổ** (window function) tính một giá trị cho **mỗi dòng** dựa trên một "cửa sổ" các dòng liên quan, **không** gộp dòng như `GROUP BY`. `DENSE_RANK() OVER (ORDER BY salary DESC)` đánh hạng theo lương giảm dần:

| Hàm | Lương `300, 300, 200, 100` | Ghi chú |
|---|---|---|
| `ROW_NUMBER()` | 1, 2, 3, 4 | luôn khác nhau |
| `RANK()` | 1, 1, 3, 4 | bằng nhau cùng hạng, **nhảy** hạng sau |
| `DENSE_RANK()` | 1, 1, 2, 3 | bằng nhau cùng hạng, **không nhảy** |

"Cao thứ `N` khác nhau" chính là các dòng có `DENSE_RANK = N`.

In [17]:
SQL_RANKS = """
SELECT salary,
       ROW_NUMBER() OVER (ORDER BY salary DESC) AS row_num,
       RANK()       OVER (ORDER BY salary DESC) AS rnk,
       DENSE_RANK() OVER (ORDER BY salary DESC) AS dense
FROM Employee
"""
print(run_sql(SQL_RANKS, {"Employee": emp_df([300, 100, 300, 200])}))

   salary  row_num  rnk  dense
0     300        1    1      1
1     300        2    1      1
2     200        3    3      2
3     100        4    4      3


In [18]:
SQL_DENSE = """
SELECT MAX(salary) AS value
FROM (
    SELECT salary, DENSE_RANK() OVER (ORDER BY salary DESC) AS dr
    FROM Employee
) AS ranked
WHERE dr = ?
"""

def nth_by_dense_rank(salaries, N):
    return rows(run_sql(SQL_DENSE, {"Employee": emp_df(salaries)}, params=(N,)))[0][0]

for sal, N in [([100, 200, 300], 2), ([100], 2), ([300, 300, 200, 100], 2), ([5, 4], 0)]:
    print(f"{str(sal):<22} N = {N}: {nth_by_dense_rank(sal, N)}")

[100, 200, 300]        N = 2: 200
[100]                  N = 2: None
[300, 300, 200, 100]   N = 2: 200
[5, 4]                 N = 0: None


`MAX(...)` gộp các dòng cùng hạng (có thể nhiều người cùng lương) thành một giá trị, và trả `NULL` khi không có dòng nào — `N <= 0` cũng tự cho `NULL`.

# 6. Cách 3: truy vấn con tương quan — "có đúng `N − 1` mức lương lớn hơn"

Mức lương `s` là cao thứ `N` khác nhau ⟺ có **đúng** `N − 1` mức lương khác nhau **lớn hơn** `s`.

```sql
SELECT DISTINCT e1.salary
FROM Employee e1
WHERE (SELECT COUNT(DISTINCT e2.salary) FROM Employee e2 WHERE e2.salary > e1.salary) = N - 1
```

Truy vấn con **tương quan** (correlated subquery) dùng giá trị `e1.salary` của dòng ngoài → chạy lại cho **mỗi dòng** → `O(n²)`. Dễ hiểu nhưng chậm với bảng lớn.

In [19]:
SQL_CORR = """
SELECT (
    SELECT DISTINCT e1.salary
    FROM Employee e1
    WHERE (SELECT COUNT(DISTINCT e2.salary) FROM Employee e2 WHERE e2.salary > e1.salary) = ?
) AS value
"""

def nth_by_correlated(salaries, N):
    return rows(run_sql(SQL_CORR, {"Employee": emp_df(salaries)}, params=(N - 1,)))[0][0]

for sal, N in [([100, 200, 300], 2), ([100], 2), ([300, 300, 200, 100], 2)]:
    print(f"{str(sal):<22} N = {N}: {nth_by_correlated(sal, N)}")

[100, 200, 300]        N = 2: 200
[100]                  N = 2: None
[300, 300, 200, 100]   N = 2: 200


# 7. Mô phỏng `DENSE_RANK`

In [20]:
def trace_dense_rank(salaries, N):
    order = sorted(salaries, reverse=True)
    rank, prev, ranks = 0, None, []
    for s in order:
        if s != prev:
            rank += 1                 # chỉ tăng hạng khi gặp lương MỚI → "dense"
            prev = s
        ranks.append(rank)
    print("lương giảm dần:", order)
    print("DENSE_RANK    :", ranks)
    hits = [s for s, r in zip(order, ranks) if r == N]
    print(f"dòng có hạng {N}:", hits, "→ MAX =", max(hits) if hits else None)

trace_dense_rank([300, 100, 300, 200, 200], 2)
trace_dense_rank([300, 100, 300, 200, 200], 4)

lương giảm dần: [300, 300, 200, 200, 100]
DENSE_RANK    : [1, 1, 2, 2, 3]
dòng có hạng 2: [200, 200] → MAX = 200
lương giảm dần: [300, 300, 200, 200, 100]
DENSE_RANK    : [1, 1, 2, 2, 3]
dòng có hạng 4: [] → MAX = None


# 8. Phiên bản nộp LeetCode

**MySQL** — xem mục 4 (cách 1). Bản dùng `DENSE_RANK` (MySQL 8+):

```sql
CREATE FUNCTION getNthHighestSalary(N INT) RETURNS INT
BEGIN
  RETURN (
      SELECT MAX(salary)
      FROM (SELECT salary, DENSE_RANK() OVER (ORDER BY salary DESC) AS dr FROM Employee) AS t
      WHERE dr = N
  );
END
```

**Pandas:**

In [21]:
def nth_highest_salary(employee: pd.DataFrame, N: int) -> pd.DataFrame:
    col = f"getNthHighestSalary({N})"
    distinct = employee["salary"].drop_duplicates().sort_values(ascending=False)
    if N <= 0 or N > len(distinct):
        return pd.DataFrame({col: [None]})
    return pd.DataFrame({col: [distinct.iloc[N - 1]]})

print(nth_highest_salary(emp_df([100, 200, 300]), 2))
print(nth_highest_salary(emp_df([100]), 2))

   getNthHighestSalary(2)
0                     200
  getNthHighestSalary(2)
0                   None


In [22]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<20} PASS  {actual}")


def expected_nth(salaries, N):
    d = sorted(set(salaries), reverse=True)
    return d[N - 1] if 1 <= N <= len(d) else None


check("Ví dụ 1 (OFFSET)", nth_by_offset([100, 200, 300], 2), 200)
check("Ví dụ 2 (OFFSET)", nth_by_offset([100], 2), None)
check("Ví dụ 1 (DENSE)", nth_by_dense_rank([100, 200, 300], 2), 200)
check("Ví dụ 1 (Pandas)", rows(nth_highest_salary(emp_df([100, 200, 300]), 2)), [[200]])
check("Tên cột Pandas", list(nth_highest_salary(emp_df([100]), 2).columns), ["getNthHighestSalary(2)"])
check("N = 1", nth_by_offset([7, 7, 3], 1), 7)

random.seed(177)
for _ in range(400):
    sal = [random.randint(1, 6) * 10 for _ in range(random.randint(1, 8))]
    N = random.randint(0, 5)
    exp = expected_nth(sal, N)
    got = [nth_by_offset(sal, N), nth_by_dense_rank(sal, N), rows(nth_highest_salary(emp_df(sal), N))[0][0]]
    if N >= 1:
        got.append(nth_by_correlated(sal, N))
    assert all(g == exp for g in got), (sal, N, got, exp)
print("400 bộ ngẫu nhiên: ba câu SQL và bản Pandas khớp đáp án tính bằng Python")

Ví dụ 1 (OFFSET)     PASS  200
Ví dụ 2 (OFFSET)     PASS  None
Ví dụ 1 (DENSE)      PASS  200
Ví dụ 1 (Pandas)     PASS  [[200]]
Tên cột Pandas       PASS  ['getNthHighestSalary(2)']
N = 1                PASS  7


400 bộ ngẫu nhiên: ba câu SQL và bản Pandas khớp đáp án tính bằng Python


# 9. Vì sao đúng? Độ phức tạp?

- **OFFSET:** dãy lương khác nhau giảm dần; bỏ `N − 1` phần tử đầu, phần tử kế là thứ `N`. Truy vấn con vô hướng biến "không có" thành `NULL`.
- **DENSE_RANK:** hạng dày đếm đúng số mức lương khác nhau lớn hơn hoặc bằng → hạng `N` ⟺ cao thứ `N` khác nhau.
- **Tương quan:** định nghĩa trực tiếp "có `N − 1` mức lớn hơn".

| Cách | Độ phức tạp điển hình |
|---|---|
| `DISTINCT` + `ORDER BY` + `OFFSET` | `O(n log n)` |
| `DENSE_RANK()` | `O(n log n)` |
| Truy vấn con tương quan | `O(n²)` |

# 10. Những lỗi dễ mắc

## Lỗi 1: `OFFSET N` thay vì `OFFSET N − 1`

In [23]:
def wrong_offset_n(salaries, N):
    return rows(run_sql(SQL_OFFSET, {"Employee": emp_df(salaries)}, params=(N,)))[0][0]   # SAI: lệch 1

print("Sai :", wrong_offset_n([100, 200, 300], 2), "| Đúng:", nth_by_offset([100, 200, 300], 2))

Sai : 100 | Đúng: 200


## Lỗi 2: Dùng `RANK()` thay vì `DENSE_RANK()`

In [24]:
SQL_RANK_WRONG = SQL_DENSE.replace("DENSE_RANK()", "RANK()")
t_rank = {"Employee": emp_df([300, 300, 200, 100])}
print("Sai :", rows(run_sql(SQL_RANK_WRONG, t_rank, params=(2,))), "| Đúng:", rows(run_sql(SQL_DENSE, t_rank, params=(2,))))

Sai : [[None]] | Đúng: [[200]]


Với `300, 300, 200`, `RANK()` cho `1, 1, 3` — không có hạng 2 → `NULL` sai. `DENSE_RANK()` cho `1, 1, 2`.

## Lỗi 3: Quên `DISTINCT` ở cách OFFSET

In [25]:
SQL_NO_DISTINCT = SQL_OFFSET.replace("SELECT DISTINCT salary", "SELECT salary")
print("Sai :", rows(run_sql(SQL_NO_DISTINCT, t_rank, params=(1,))), "| Đúng:", rows(run_sql(SQL_OFFSET, t_rank, params=(1,))))

Sai : [[300]] | Đúng: [[200]]


## Lỗi 4: Viết `LIMIT 1 OFFSET N - 1` trong MySQL

MySQL báo **lỗi cú pháp** vì `LIMIT`/`OFFSET` không nhận biểu thức. Cần `SET N = N - 1;` trước `RETURN`. (SQLite thì chấp nhận biểu thức — một khác biệt giữa các hệ CSDL cần lưu ý.)

# 11. Mẹo nhận diện pattern

- **"Thứ N", "top N", "xếp hạng"** → hàm cửa sổ `ROW_NUMBER` / `RANK` / `DENSE_RANK`; chọn đúng loại theo cách xử lý **bằng nhau**.
- **Cần một giá trị, có thể không tồn tại** → truy vấn con vô hướng hoặc `MAX`/`MIN` (trả `NULL` trên tập rỗng).
- **Hàm có tham số** → nhớ giới hạn cú pháp của hệ CSDL (MySQL `LIMIT` chỉ nhận hằng/biến).

Bài liên quan: **#176 Second Highest Salary**, **#178 Rank Scores**, **#185 Department Top Three Salaries** (Hard), **#1077**, **#2668**.

# 12. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Với lương `[50, 80, 80, 20, 50, 90]`, viết `ROW_NUMBER`, `RANK`, `DENSE_RANK` theo lương giảm dần, và cho biết lương cao thứ 3 khác nhau.

## Bài 2 — Viết câu SQL cách OFFSET (tham số `?` là `N − 1`)

In [26]:
def exercise_2_sql() -> str:
    # TODO: trả về chuỗi SQL có một tham số ? cho OFFSET
    pass


# assert rows(run_sql(exercise_2_sql(), {"Employee": emp_df([100, 200, 300])}, params=(1,))) == [[200]]
# assert rows(run_sql(exercise_2_sql(), {"Employee": emp_df([100])}, params=(1,))) == [[None]]

## Bài 3 — Viết câu SQL cách `DENSE_RANK` (tham số `?` là `N`)

In [27]:
def exercise_3_sql() -> str:
    # TODO
    pass


# assert rows(run_sql(exercise_3_sql(), {"Employee": emp_df([300, 300, 200, 100])}, params=(2,))) == [[200]]
# assert rows(run_sql(exercise_3_sql(), {"Employee": emp_df([300])}, params=(2,))) == [[None]]

## Bài 4 — Biến thể: liệt kê **mọi nhân viên** có lương cao thứ `N` (cột `id`, `salary`, sắp theo `id`)

In [28]:
def exercise_4_sql() -> str:
    # TODO
    pass


# assert rows(run_sql(exercise_4_sql(), {"Employee": emp_df([300, 200, 300, 200, 100])}, params=(2,))) == [[2, 200], [4, 200]]
# assert rows(run_sql(exercise_4_sql(), {"Employee": emp_df([1])}, params=(3,))) == []

## Bài 5 — Mini challenge: bản Pandas dùng `rank(method="dense")`

`Series.rank(method="dense", ascending=False)` cho hạng dày như `DENSE_RANK`. Viết lại `nth_highest_salary` bằng cách này.

In [29]:
def exercise_5_nth_by_rank(employee: pd.DataFrame, N: int) -> pd.DataFrame:
    # TODO
    pass


# assert rows(exercise_5_nth_by_rank(emp_df([100, 200, 300]), 2)) == [[200]]
# assert rows(exercise_5_nth_by_rank(emp_df([300, 300, 100]), 2)) == [[100]]
# assert rows(exercise_5_nth_by_rank(emp_df([100]), 2)) == [[None]] and list(exercise_5_nth_by_rank(emp_df([100]), 2).columns) == ["getNthHighestSalary(2)"]

# 13. Gợi ý

**Bài 1**
- Mức 1: chạy `run_sql(SQL_RANKS, {"Employee": emp_df([...])})` và `trace_dense_rank` sau khi tự làm.

**Bài 4**
- Mức 1: tính `DENSE_RANK` trong truy vấn con, ngoài lọc `WHERE dr = ?` và `ORDER BY id`.

**Bài 5**
- Mức 1: `r = employee["salary"].rank(method="dense", ascending=False)`; lọc `employee["salary"][r == N]`; rỗng thì `None`, ngược lại lấy `max()`.

# 14. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [30]:
print(run_sql(SQL_RANKS, {"Employee": emp_df([50, 80, 80, 20, 50, 90])}))
trace_dense_rank([50, 80, 80, 20, 50, 90], 3)

   salary  row_num  rnk  dense
0      90        1    1      1
1      80        2    2      2
2      80        3    2      2
3      50        4    4      3
4      50        5    4      3
5      20        6    6      4
lương giảm dần: [90, 80, 80, 50, 50, 20]
DENSE_RANK    : [1, 2, 2, 3, 3, 4]
dòng có hạng 3: [50, 50] → MAX = 50


## Đáp án Bài 2

In [31]:
def answer_2_sql() -> str:
    return SQL_OFFSET

assert rows(run_sql(answer_2_sql(), {"Employee": emp_df([100, 200, 300])}, params=(1,))) == [[200]]
assert rows(run_sql(answer_2_sql(), {"Employee": emp_df([100])}, params=(1,))) == [[None]]
print("Bài 2 PASS (giống SQL_OFFSET ở mục 4)")

Bài 2 PASS (giống SQL_OFFSET ở mục 4)


## Đáp án Bài 3

In [32]:
def answer_3_sql() -> str:
    return SQL_DENSE

assert rows(run_sql(answer_3_sql(), {"Employee": emp_df([300, 300, 200, 100])}, params=(2,))) == [[200]]
assert rows(run_sql(answer_3_sql(), {"Employee": emp_df([300])}, params=(2,))) == [[None]]
print("Bài 3 PASS (giống SQL_DENSE ở mục 5)")

Bài 3 PASS (giống SQL_DENSE ở mục 5)


## Đáp án Bài 4

In [33]:
def answer_4_sql() -> str:
    return """
    SELECT id, salary
    FROM (SELECT id, salary, DENSE_RANK() OVER (ORDER BY salary DESC) AS dr FROM Employee) AS t
    WHERE dr = ?
    ORDER BY id
    """

assert rows(run_sql(answer_4_sql(), {"Employee": emp_df([300, 200, 300, 200, 100])}, params=(2,))) == [[2, 200], [4, 200]]
assert rows(run_sql(answer_4_sql(), {"Employee": emp_df([1])}, params=(3,))) == []
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [34]:
def answer_5_nth_by_rank(employee: pd.DataFrame, N: int) -> pd.DataFrame:
    col = f"getNthHighestSalary({N})"
    r = employee["salary"].rank(method="dense", ascending=False)
    hit = employee["salary"][r == N]
    return pd.DataFrame({col: [hit.max() if len(hit) else None]})

assert rows(answer_5_nth_by_rank(emp_df([100, 200, 300]), 2)) == [[200]]
assert rows(answer_5_nth_by_rank(emp_df([300, 300, 100]), 2)) == [[100]]
assert rows(answer_5_nth_by_rank(emp_df([100]), 2)) == [[None]] and list(answer_5_nth_by_rank(emp_df([100]), 2).columns) == ["getNthHighestSalary(2)"]
for _ in range(300):
    sal = [random.randint(1, 5) for _ in range(random.randint(1, 7))]
    N = random.randint(0, 5)
    assert rows(answer_5_nth_by_rank(emp_df(sal), N))[0][0] == expected_nth(sal, N)
print("Bài 5 PASS")

Bài 5 PASS


# 15. Tổng kết

> Lương cao thứ `N` khác nhau: `SELECT DISTINCT salary … ORDER BY salary DESC LIMIT 1 OFFSET N−1` bọc trong truy vấn con (MySQL: `SET N = N − 1` trước vì `OFFSET` không nhận biểu thức), hoặc `MAX(salary) WHERE DENSE_RANK() = N`. Chọn `DENSE_RANK` (không nhảy hạng), không phải `RANK`.

---

## Checklist trước khi nộp

- [ ] `OFFSET N − 1` (đã `SET N = N − 1` trong MySQL)?
- [ ] Có `DISTINCT` (hoặc dùng `DENSE_RANK`)?
- [ ] Không đủ `N` mức → `NULL`?
- [ ] Bản Pandas: tên cột `getNthHighestSalary(N)`?